# Trabalho Final - Parte 2 - Parcial



## Processos em Python

### GIL do Python

O GIL (Global Interpreter Lock) é um bloqueio global que garante que apenas uma thread de Python bytecode esteja executando por vez dentro de um processo.

Isso significa que, mesmo criando várias threads, apenas uma thread consegue rodar instruções Python de cada vez.

Ou seja, por padrão threads em Python não aceleram tarefas CPU-bound.

### Multithreading e Multiprocessing

É possível criar múltiplas threads em Python através do módulo `threading` dentro de um mesmo processo Python, porém segue válida a limitação de executar apenas uma thread por vez dentro do mesmo processador, ou seja, as threads realizam chaveamento entre si em um processador, sem de fato executar em paralelo.

```python
import threading
import time

def tarefa(nome):
    for i in range(3):
        print(f"Tarefa {nome}: {i}")
        time.sleep(1)

# Criando duas threads
t1 = threading.Thread(target=tarefa, args=("A",))
t2 = threading.Thread(target=tarefa, args=("B",))

t1.start()
t2.start()

t1.join()
t2.join()
```


Com o objetivo de utilizar **múltiplos núcleos de CPU**, Python possui o módulo `multiprocessing`. Esse módulo cria **processos independentes** cada um com seu próprio interpretador e sua própria memória, evitando o GIL. Como resultado várias tarefas podem efetivamente executar em paralelo.

```python
from multiprocessing import Process
import math

def calcular():
    print(sum(i*i for i in range(10_000_000)))

processos = [Process(target=calcular) for _ in range(4)]

for p in processos:
    p.start()
for p in processos:
    p.join()
```

## Identificando IDs de núcleos do sistema

Um comando bastante comum para identificar os IDs dos núcleos de CPU disponíveis é o `lscpu`. Para visualizar apenas o ID dos núcleos pode-se utilziar `lscpu -p=CPU`. Alternativamente, `lscpu --extended` mostra, além dos IDs, mais detalhes sobre os núcleos, tal como número do socket de CPU, tamanho de memória cache e frequências de operação dos núcleos.

In [ ]:
!lscpu -p=CPU

In [ ]:
!lscpu --extended

## Identificando IDs de núcleos do sistema (com Python)

Outra forma é verificar com o módulo `psutil`, utilziando Python puro:

```python
import psutil
psutil.cpu_count(logical=True)   # Número de CPUs lógicas
psutil.cpu_count(logical=False)  # Número de núcleos físicos
```

In [ ]:
# Verificando IDs dos núcleos do sistema com Python puro (módulo psutil)

import psutil
print("Número de núcleos lógicos: " + str(psutil.cpu_count(logical=True)))
print("Número de núcleos físicos: " + str(psutil.cpu_count(logical=False)))

print("IDs lógicos disponíveis: " + str(list(range(psutil.cpu_count(logical=True)))))

## Afinidade de CPU (CPU affinity)

Por padrão o kernel do Linux é livre para mover um processo entre quaisquer núcleos de CPU disponíveis no sistema. No Ubuntu (Linux), entretanto, é possível controlar em qual CPU cada processo da aplicação Python irá executar com o uso da **afinidade de CPU**. 

Cada processo no Linux pode ter uma **máscara de afinidade** indicando quais núcleos ele está autorizado a executar.

### Comando os.sched_setaffinity()

Em Python, no Linux, a afinidade pode ser definidade utilizando `os.sched_setaffinity`. A chamada de sistema é utilizada diretamente no código:

```python
import os
import multiprocessing

def tarefa(cpu_id):
    # Define afinidade para o processo atual
    pid = os.getpid()
    os.sched_setaffinity(pid, {cpu_id})
    print(f"Processo {pid} rodando no CPU {cpu_id}")
    while True:
        pass  # loop infinito para ocupar CPU

if __name__ == "__main__":
    processos = []
    for cpu in range(2):  # usar apenas CPU 0 e CPU 1
        p = multiprocessing.Process(target=tarefa, args=(cpu,))
        processos.append(p)
        p.start()

    for p in processos:
        p.join()
```


In [ ]:
# Utilizando os.sched_setaffinity (Linux-only)
# Execução de processos em núcleos definidos

import os
import multiprocessing

def tarefa(cpu_id):
    # Define afinidade para o processo atual
    pid = os.getpid()
    os.sched_setaffinity(pid, {cpu_id})
    print(f"Processo {pid} rodando no CPU {cpu_id}")
    while True:
        pass  # loop infinito para ocupar CPU

if __name__ == "__main__":
    processos = []
    for cpu in range(2):  # usar apenas CPU 0 e CPU 1
        p = multiprocessing.Process(target=tarefa, args=(cpu,))
        processos.append(p)
        p.start()

    for p in processos:
        p.join()

### Utilizando `psutil` (multiplataforma)

O módulo `psutil`, diferentemente do comando `os.sched_setaffinity()` oferece portabilidade por ser multiplataforma, isto é, não dependente do Linux.

```python
import os
import psutil
import multiprocessing

def tarefa(cpu_id):
    pid = os.getpid()
    p = psutil.Process(pid)
    p.cpu_affinity([cpu_id])  # define a afinidade
    print(f"Processo {pid} rodando no CPU {cpu_id}")
    while True:
        pass

if __name__ == "__main__":
    processos = []
    for cpu in [0, 1, 2, 3]:  # escolher CPUs manualmente
        p = multiprocessing.Process(target=tarefa, args=(cpu,))
        processos.append(p)
        p.start()

    for p in processos:
        p.join()
```

Pode ser necessário instalar o `psutil` no sistema: 

```bash
sudo apt install python3-psutil
```

In [ ]:
# Utilizando psutil

cpu_ids = list(range(psutil.cpu_count(logical=True)))

import os
import psutil
import multiprocessing

def tarefa(cpu_id):
    pid = os.getpid()
    p = psutil.Process(pid)
    p.cpu_affinity([cpu_id])  # define a afinidade
    print(f"Processo {pid} rodando no CPU {cpu_id}")
    while True:
        pass

if __name__ == "__main__":
    processos = []
    #for cpu in [0, 1, 2, 3]:  # escolher núcleos manualmente
    for cpu in cpu_ids:  
        p = multiprocessing.Process(target=tarefa, args=(cpu,))
        processos.append(p)
        p.start()

    for p in processos:
        p.join()

## CPU governor

O **CPU governor** é uma política de escalonamento de frequência que o kernel do Linux aplica sobre os núcleos do processador. Ele é apenas uma camada de política, ou seja, um conjunto de regras de como usar a CPU. 

O governor decide **em qual frequência a CPU deve operar em um dado momento**, de acordo com critérios como:

- **Carga de trabalho** (se o sistema está ocioso ou ocupado).
- **Economia de energia** (modo portátil, servidor de baixo consumo).
- **Desempenho máximo** (quando a prioridade é velocidade).


### Principais tipos de governors

Dependendo do driver e da CPU, diferentes governors ficam disponíveis. Os mais comuns são:

**1. performance**
- Mantém a CPU sempre na frequência máxima.
- Vantagem: maior desempenho, útil para cargas pesadas e baixa latência.
- Desvantagem: maior consumo de energia e calor.
- Exemplo: ideal para servidores de banco de dados ou aplicações críticas.

**2. powersave**
- Mantém a CPU sempre na frequência mínima.
- Vantagem: reduz o consumo de energia.
- Desvantagem: desempenho limitado.
- Exemplo: bom para servidores de background ou quando o foco é economia.

**3. ondemand**
- Ajusta dinamicamente a frequência de acordo com a carga.
- Se a carga aumenta, sobe para máxima; se cai, reduz para mínima.
- Era muito usado em kernels antigos.

**4. conservative**
- Similar ao ondemand, mas aumenta/diminui a frequência de forma mais gradual.
- Útil para economizar energia sem oscilações bruscas.

**5. schedutil (moderno)**
- Integra o ajuste de frequência diretamente com o escalonador do kernel.
- Usa informações de agendamento de processos para decidir a frequência.
- É o padrão em muitas distros Linux recentes.


### Descobrindo os governos disponíveis

Comando:
```bash
cat /sys/devices/system/cpu/cpu0/cpufreq/scaling_available_governors
```

Saída:
```bash
performance powersave
```


In [ ]:
# Verificando governors disponíveis
!cat /sys/devices/system/cpu/cpu0/cpufreq/scaling_available_governors

### Verificando o governor de cada núcleo de CPU


Comando:
```bash
cat /sys/devices/system/cpu/cpu*/cpufreq/scaling_governor
```

Saída:
```bash
powersave
powersave
powersave
powersave
powersave
powersave
powersave
powersave
```

Obs.: Considerando que há 8 núcleos disponíveis, isto é, CPU0 a CPU7

In [ ]:
%%bash
# Governor de todas as CPUs

for cpu in /sys/devices/system/cpu/cpu*/cpufreq/scaling_governor; do
    num=$(echo $cpu | grep -oP 'cpu\K[0-9]+')
    echo "CPU $num: $(cat $cpu)"
done

### Alterar Governor

Alteração:
```bash
for cpu in /sys/devices/system/cpu/cpu*/cpufreq/scaling_governor; do
    echo performance | sudo tee $cpu
done
```

Verificação:
```bash
cat /sys/devices/system/cpu/cpu*/cpufreq/scaling_governor
```

In [ ]:
# Alterando o governor: Código em Python para receber senha de sudo e executar alterações

import getpass
import os

# Pede a senha de sudo sem exibir na tela
password = getpass.getpass("Sudo password: ")

# Comando em bash que altera o governor para "performance"
bash_command = """
for cpu in /sys/devices/system/cpu/cpu*/cpufreq/scaling_governor; do
    echo performance | sudo tee $cpu
done
"""

# Monta o comando com sudo -S (lê senha do stdin)
command = f'echo "{password}" | sudo -S bash -c \'{bash_command}\''
os.system(command)

In [ ]:
%%bash
# Verificação final do governor, após modificação
cat /sys/devices/system/cpu/cpu*/cpufreq/scaling_governor

## Verificando o driver da CPU

O **driver** da CPU é diferente do governor. Quem "fala" com o hardware é o driver, como `intel_pstate` ou `acpi-cpufreq`.

Comando:
```bash
cat /sys/devices/system/cpu/cpu*/cpufreq/scaling_driver
```

Saída:
```bash
intel_pstate
intel_pstate
intel_pstate
intel_pstate
intel_pstate
intel_pstate
intel_pstate
intel_pstate
```

Obs.: Considerando que há 8 núcleos disponíveis, isto é, CPU0 a CPU7

In [ ]:
%%bash
# Driver de cada CPU

for cpu in /sys/devices/system/cpu/cpu*/cpufreq/scaling_driver; do
    num=$(echo $cpu | grep -oP 'cpu\K[0-9]+')
    echo "CPU $num: $(cat $cpu)"
done

## Verificando a frequência de operação dos núcleos

## Cargas de trabalho CPU-bound

- Criptografia
- Multiplicação de Matrizes

# Criptografia

- AES-CTR

Aplciação precisa:
- Definir governor das CPUs para ```performance```
- Verificar driver da CPU
- verificar range de frequencia de operação
- Definir um processo para cada núcleo

In [ ]:
#!pip install pycryptodome

In [ ]:
import sys
print(sys.executable)

In [ ]:
import Cryptodome
print(Cryptodome.__version__)

In [ ]:
# Parallel AES-CTR Benchmark Script (PyCryptodome)

import os
import time
from Cryptodome.Cipher import AES
from concurrent.futures import ThreadPoolExecutor

# Parameters
KEY = os.urandom(32)   # AES-256 key
NONCE = os.urandom(8)  # 64-bit nonce
THREADS = 8
BLOCK_SIZE = 16  # AES block size in bytes

def encrypt_chunk(chunk, counter_start):
    """Encrypt one chunk with AES-CTR using a thread-specific counter offset."""
    cipher = AES.new(KEY, AES.MODE_CTR, nonce=NONCE, initial_value=counter_start)
    return cipher.encrypt(chunk)

def parallel_encrypt(data, num_threads=THREADS):
    """Encrypt data in parallel with exactly num_threads threads."""
    chunk_size = len(data) // num_threads
    chunks = [data[i*chunk_size:(i+1)*chunk_size] for i in range(num_threads)]
    
    # Assign counters so keystreams don't overlap
    counters = [i * (chunk_size // BLOCK_SIZE) for i in range(num_threads)]
    
    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        encrypted_chunks = list(executor.map(encrypt_chunk, chunks, counters))
    
    return b''.join(encrypted_chunks)

def run_benchmark(data_size_bytes):
    """Run encryption benchmark for a given data size (bytes)."""
    print(f"\n=== Benchmark: {data_size_bytes / (1024*1024)} MB ===")
    data = os.urandom(data_size_bytes)  # Allocate test data
    
    start = time.perf_counter()
    ciphertext = parallel_encrypt(data)
    end = time.perf_counter()
    
    elapsed = end - start
    throughput_mb_s = (data_size_bytes / (1024*1024)) / elapsed
    
    print(f"Time: {elapsed:.2f} s")
    print(f"Throughput: {throughput_mb_s:.2f} MB/s using {THREADS} threads")

if __name__ == "__main__":
    sizes = [
        512 * 1024 * 1024,   # 512 MB
        1024 * 1024 * 1024,  # 1 GB
        2 * 1024 * 1024 * 1024  # 2 GB
    ]
    for size in sizes:
        run_benchmark(size)

In [ ]:
# Correctness Check

from Cryptodome.Cipher import AES

def decrypt_chunk(chunk, counter_start):
    cipher = AES.new(KEY, AES.MODE_CTR, nonce=NONCE, initial_value=counter_start)
    return cipher.decrypt(chunk)

def parallel_decrypt(ciphertext, num_threads=THREADS):
    chunk_size = len(ciphertext) // num_threads
    chunks = []
    start = 0
    for i in range(num_threads):
        end = start + chunk_size
        if i == num_threads - 1:
            end = len(ciphertext)
        chunks.append(ciphertext[start:end])
        start = end

    blocks_per_chunk = (len(ciphertext) + num_threads - 1) // num_threads // BLOCK_SIZE
    counters = [i * blocks_per_chunk for i in range(num_threads)]

    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        decrypted_chunks = list(executor.map(decrypt_chunk, chunks, counters))

    return b''.join(decrypted_chunks)

data_size_bytes = 512 * 1024 * 1024 # 512 MB
data = os.urandom(data_size_bytes)
ciphertext = parallel_encrypt(data)
recovered = parallel_decrypt(ciphertext)
assert recovered == data, "Decryption failed! Data does not match."
print("Encryption verified successfully!")

In [ ]:
# Install only core parts of dask
!pip install "dask[complete]"

# Multiplicação de Matrizes

In [ ]:
"""
Safe Dask Matrix Multiplication Script
--------------------------------------
- MATRIX_SIZE is configurable (default 8000)
- NUM_WORKERS sets exactly 8 parallel Dask worker processes
- THREADS_PER_WORKER = 1 for BLAS / NumPy
- Before computing, checks that MATRIX_SIZE is divisible by NUM_WORKERS
"""

import os
import numpy as np
import dask.array as da
from dask.distributed import Client
import time
import sys

# -----------------------------
# CONFIGURATION
MATRIX_SIZE = 16000       # Size of NxN matrices
NUM_WORKERS = 8          # Number of Dask worker processes
THREADS_PER_WORKER = 1   # Threads per worker (BLAS / NumPy)

# -----------------------------
# Safety check: ensure MATRIX_SIZE is divisible by NUM_WORKERS
if MATRIX_SIZE % NUM_WORKERS != 0:
    print(f"Error: MATRIX_SIZE ({MATRIX_SIZE}) is not divisible by NUM_WORKERS ({NUM_WORKERS}).")
    print("Please choose a MATRIX_SIZE divisible by NUM_WORKERS.")
    sys.exit(1)

# -----------------------------
# Set BLAS threading to 1 per worker to avoid oversubscription
os.environ["OMP_NUM_THREADS"] = str(THREADS_PER_WORKER)
os.environ["MKL_NUM_THREADS"] = str(THREADS_PER_WORKER)
os.environ["OPENBLAS_NUM_THREADS"] = str(THREADS_PER_WORKER)

# -----------------------------
# Start Dask client
client = Client(n_workers=NUM_WORKERS, threads_per_worker=THREADS_PER_WORKER)
print(f"Dask client started with {NUM_WORKERS} workers")

# -----------------------------
# Define chunk sizes
chunk_rows = MATRIX_SIZE // NUM_WORKERS
chunk_cols = MATRIX_SIZE // NUM_WORKERS

# Create Dask arrays
A = da.random.random((MATRIX_SIZE, MATRIX_SIZE), chunks=(chunk_rows, MATRIX_SIZE))
B = da.random.random((MATRIX_SIZE, MATRIX_SIZE), chunks=(MATRIX_SIZE, chunk_cols))

# Print chunk sizes for verification
print(f"A chunks: {A.chunks}")
print(f"B chunks: {B.chunks}")

# -----------------------------
# Matrix multiplication
start_time = time.time()
C = A @ B
result = C.compute()  # Trigger parallel execution
end_time = time.time()

print("Matrix multiplication completed")
print(f"Result shape: {result.shape}")
print(f"Elapsed time: {end_time - start_time:.2f} seconds")

# -----------------------------
# Clean up
client.close()
